# Agent evaluation

End-to-end evaluation of the RAG agent across four models. The flow is:

1. **Generate answers** — run each model over the ground-truth questions and save its answers + tool calls.
2. **Judge** — an LLM-as-a-judge scores every answer on two axes: `answer_score` (is the final answer correct?) and `trajectory_score` (were the tool calls reasonable?).
3. **Compare** — aggregate good-rates per model.
4. **Analyze** — inspect the bad answers and bad tool trajectories to understand the trade-offs.

In [ ]:
import sys
sys.path.append("..")
import json
from pathlib import Path
from typing import Literal

import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv

from rag import RAG

load_dotenv()

True

In [8]:
df_ground_truth = pd.read_csv("ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:

def generate_agent_answer(rag, rec):
    question = rec["question"]
    answer_agent = rag.rag(question)
    json_file = Path("samples") / rec["filename"]
    with json_file.open("r", encoding="utf-8") as f:
        data = json.load(f)
        content = data["content"]

    return {
        "question": rec["question"],
        "title": rec["title"],
        "answer_agent": answer_agent,
        "tool_calls": json.dumps(rag.tool_calls),
        "filename": rec["filename"],
        "content": content,
    }

## Generate answers for each model

In [6]:
model = "gpt-5.4-mini"
Rag = RAG(model=model, provider="openai")
results = []
for rec in tqdm(ground_truth):
    results.append(generate_agent_answer(Rag, rec))
df_answers = pd.DataFrame(results)
df_answers.to_csv(f"agent-answers-{model}.csv", index=False)

  0%|          | 0/55 [00:00<?, ?it/s]

In [7]:
model = "gpt-5-nano-2025-08-07"
Rag = RAG(model=model, provider="openai")
results = []
for rec in tqdm(ground_truth):
    results.append(generate_agent_answer(Rag, rec))
df_answers = pd.DataFrame(results)
df_answers.to_csv(f"agent-answers-{model}.csv", index=False)

  0%|          | 0/55 [00:00<?, ?it/s]

In [8]:
model = "deepseek-v4-flash"
Rag = RAG(model=model, provider="deepseek")
results = []
for rec in tqdm(ground_truth):
    results.append(generate_agent_answer(Rag, rec))
df_answers = pd.DataFrame(results)
df_answers.to_csv(f"agent-answers-{model}.csv", index=False)

  0%|          | 0/55 [00:00<?, ?it/s]

In [9]:
model = "deepseek-v4-pro"
Rag = RAG(model=model, provider="deepseek")
results = []
for rec in tqdm(ground_truth):
    results.append(generate_agent_answer(Rag, rec))
df_answers = pd.DataFrame(results)
df_answers.to_csv(f"agent-answers-{model}.csv", index=False)

  0%|          | 0/55 [00:00<?, ?it/s]

## Define the LLM-as-a-judge

The judge returns a structured verdict on both the final answer and the tool-call trajectory.

In [38]:
class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [39]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.

Return a JSON object with exactly these four fields:
- answer_reasoning: your written reasoning about the answer quality
- answer_score: "good" or "bad"
- trajectory_reasoning: your written reasoning about the trajectory quality
- trajectory_score: "good" or "bad"
""".strip()

agent_judge_prompt = """
Question:
{question}

Original content (ground truth):
{content}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()


In [45]:
def evaluate(question, content, answer_agent, tool_calls):
    prompt = agent_judge_prompt.format(
        question=question,
        content=content,
        answer_agent=answer_agent,
        tool_calls=tool_calls,
    )
    llm = ChatDeepSeek(model="deepseek-v4-pro")
    structured_llm = llm.with_structured_output(AgentEvaluation, include_raw=True, method="json_mode")
    response = structured_llm.invoke(
        [
            ("system", agent_judge_instructions),
            ("user", prompt),
        ]
    )
    answer = response["parsed"]

    return {
        "question": question,
        "answer_agent": answer_agent,
        "answer_score": answer.answer_score,
        "answer_reasoning": answer.answer_reasoning,
        "trajectory_score": answer.trajectory_score,
        "trajectory_reasoning": answer.trajectory_reasoning,
    }


## Use judge to evaluate the answers generated by different models

In [47]:
def judge_agent_answers(model):
    df_answers = pd.read_csv(f"agent-answers-{model}.csv")
    answers = df_answers.to_dict(orient="records")
    agent_judge_evals = []
    for rec in tqdm(answers):
        eval_result = evaluate(
            question=rec["question"],
            content=rec["content"],
            answer_agent=rec["answer_agent"],
            tool_calls=rec["tool_calls"]
        )
        agent_judge_evals.append(eval_result)

    df_eval = pd.DataFrame(agent_judge_evals)
    df_eval.to_csv(f"agent-evaluation-{model}.csv", index=False)

In [48]:
models = ["gpt-5.4-mini", "gpt-5-nano-2025-08-07", "deepseek-v4-flash", "deepseek-v4-pro"]
for model in models:
    judge_agent_answers(model)

  0%|          | 0/55 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

## Compare model scores

Aggregate the judge verdicts into a good-rate per model for both answer quality and tool-call quality.

In [11]:
def judge_agent_scores(df_eval):
    good_count = (df_eval["answer_score"] == "good").sum()
    total_count = len(df_eval)
    good_tool_count = (df_eval["trajectory_score"] == "good").sum()
    return good_count/total_count, good_tool_count/total_count

In [12]:
models = ["gpt-5.4-mini", "gpt-5-nano-2025-08-07", "deepseek-v4-flash", "deepseek-v4-pro"]
model_scores = {}
for model in models:
    df = pd.read_csv(f"agent-evaluation-{model}.csv")
    answer_score, tool_score = judge_agent_scores(df)
    model_scores[model] = (round(answer_score, 2), round(tool_score, 2))

pd.DataFrame(
    {"model": model_scores.keys(), "answer_score": [score[0] for score in model_scores.values()], "tool_score": [score[1] for score in model_scores.values()]}
).sort_values("answer_score", ascending=False).reset_index(drop=True)


,model,answer_score,tool_score
0,deepseek-v4-flash,0.98,0.85
1,deepseek-v4-pro,0.95,0.89
2,gpt-5.4-mini,0.87,0.98
3,gpt-5-nano-2025-08-07,0.85,1.00


`deepseek-v4-flash` has the best answer score and the lowest tool score. `gpt-5-nano-2025-08-07` has the lowest answer score and the highest tool score. This suggests that `deepseek-v4-flash` is the best model for generating answers in this evaluation.

## Analysis of bad answers

In [ ]:
df_flash_eval = pd.read_csv("agent-evaluation-deepseek-v4-flash.csv")
df_bad = df_flash_eval[df_flash_eval["answer_score"] == "bad"].head()
df_bad

,question,answer_agent,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
6,2. Why does Byzantine failure handling need mo...,"Based on the papers in the database, here is t...",bad,The agent's final answer gives consensus bound...,good,The search queries are relevant and contain im...


In [ ]:
df_deepseek_pro_eval = pd.read_csv("agent-evaluation-deepseek-v4-pro.csv")
df_bad = df_deepseek_pro_eval[df_deepseek_pro_eval["answer_score"] == "bad"].head()
df_bad

,question,answer_agent,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
6,2. Why does Byzantine failure handling need mo...,"Based on the papers in the database, the reaso...",bad,The agent's answer is not consistent with the ...,bad,The searches are topically related to Byzantin...
40,1. Why does logistic-loss boosting help reduce...,Based on the paper describing **LogisticRank**...,bad,The original answer provided is only the paper...,good,The two search queries are relevant and includ...
49,5. How does shrinking help isolate minimal fai...,Shrinking isolates minimal failing command seq...,bad,The agent answer captures part of the key info...,good,The two search queries are relevant to the que...


In [ ]:
df_gpt_mini_eval = pd.read_csv("agent-evaluation-gpt-5.4-mini.csv")
df_bad = df_gpt_mini_eval[df_gpt_mini_eval["answer_score"] == "bad"].head()
df_bad

,question,answer_agent,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
4,5. How do contract creation and message calls ...,According to the Ethereum paper:\n\n- **Contra...,bad,The agent's answer correctly explains contract...,good,"The agent performed two searches, both contain..."
8,4. When can request ordering be relaxed becaus...,Request ordering can be relaxed when the opera...,bad,The agent correctly states that request orderi...,bad,The first search query contains relevant keywo...
39,5. Why does the dining philosophers example st...,The dining philosophers example still allows s...,bad,The agent answer does not match the original. ...,good,The single search query is relevant and includ...
40,1. Why does logistic-loss boosting help reduce...,Logistic-loss boosting helps reduce bad top re...,bad,The provided ground-truth answer is the abstra...,good,The agent made two search calls. The first que...
41,2. How do contextual reranking features like r...,"After the first ranking stage, contextual rera...",bad,The original provided ground truth is only the...,good,The two search queries are relevant and includ...


In [ ]:
df_gpt_nano_eval = pd.read_csv("agent-evaluation-gpt-5-nano-2025-08-07.csv")
df_bad = df_gpt_nano_eval[df_gpt_nano_eval["answer_score"] == "bad"].head()
df_bad

,question,answer_agent,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
6,2. Why does Byzantine failure handling need mo...,Short answer:\nBecause Byzantine faults can mi...,bad,The agent's answer provides widely known Byzan...,good,The two search queries are relevant and includ...
29,5. When are blocking reclamation methods actua...,Short answer: Blocking reclamation methods are...,bad,The answer correctly states that no reclamatio...,good,The agent made a single search call with a rel...
39,5. Why does the dining philosophers example st...,- Why starvation can occur in the dining philo...,bad,The agent answer does not match the key inform...,good,The search query is directly relevant and cont...
40,1. Why does logistic-loss boosting help reduce...,Short answer:\n\n- Why logistic-loss boosting ...,bad,The original ground truth consists only of the...,good,The agent made a single search query that incl...
41,2. How do contextual reranking features like r...,"From the Yahoo contextual reranking work, cont...",bad,The original ground truth provided is only the...,good,The agent made a single search query that cont...


Answer quality is inversely related to the tool score: high tool scores tend to come from a single concise query, whereas bad answers often involve multiple, broader queries that the judge penalizes.

## Analysis of bad tool trajectories

In [ ]:
df_flash_eval = pd.read_csv("agent-evaluation-deepseek-v4-flash.csv")
df_bad_tool = df_flash_eval[df_flash_eval["trajectory_score"] == "bad"].head()
df_bad_tool

,question,answer_agent,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
4,5. How do contract creation and message calls ...,Based on the Ethereum yellow paper (*Ethereum:...,good,The final answer correctly covers the three re...,bad,The search queries are relevant and include im...
9,5. How do reconfiguration and repaired compone...,"Based on the papers in the database, here's ho...",good,The agent's final answer captures the key poin...,bad,The search queries are all relevant and includ...
13,4. How does choosing different source distribu...,Based on the papers in my database (primarily ...,good,The final answer correctly explains that chang...,bad,The search queries are relevant and include im...
26,2. What makes epoch-based reclamation easier t...,"Based on the paper database, the key source he...",good,The agent's answer correctly identifies that E...,bad,The search queries are relevant and contain im...
35,1. Why are input and output treated as primiti...,"Based on the papers in my database, this quest...",good,The agent answer is correct and aligns with th...,bad,The search queries were relevant and included ...


In [18]:
df_deepseek_pro_eval = pd.read_csv(f"agent-evaluation-deepseek-v4-pro.csv")
df_bad_tool = df_deepseek_pro_eval[df_deepseek_pro_eval["trajectory_score"] == "bad"].head()
df_bad_tool

,question,answer_agent,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,"1. How do blocks, transactions, and state upda...",Based on the Ethereum execution-layer specific...,good,The agent answer correctly explains Ethereum a...,bad,The searches are relevant and include importan...
4,5. How do contract creation and message calls ...,Based on the Ethereum yellow paper in the data...,good,The agent answer accurately captures the key d...,bad,The tool calls used relevant keywords from the...
6,2. Why does Byzantine failure handling need mo...,"Based on the papers in the database, the reaso...",bad,The agent's answer is not consistent with the ...,bad,The searches are topically related to Byzantin...
9,5. How do reconfiguration and repaired compone...,"Based on the database, this question is answer...",good,The agent answer correctly addresses how recon...,bad,The search queries are topically relevant and ...
32,3. How does the fast-path/slow-path design gua...,"Based on the wait-free queue paper (""A Wait-fr...",good,The final answer correctly explains the fast-p...,bad,The first three search queries are relevant an...


In [19]:
df_gpt_mini_eval = pd.read_csv(f"agent-evaluation-gpt-5.4-mini.csv")
df_bad_tool = df_gpt_mini_eval[df_gpt_mini_eval["trajectory_score"] == "bad"].head()
df_bad_tool

,question,answer_agent,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
8,4. When can request ordering be relaxed becaus...,Request ordering can be relaxed when the opera...,bad,The agent correctly states that request orderi...,bad,The first search query contains relevant keywo...


In [20]:
df_gpt_nano_eval = pd.read_csv(f"agent-evaluation-gpt-5-nano-2025-08-07.csv")
df_bad_tool = df_gpt_nano_eval[df_gpt_nano_eval["trajectory_score"] == "bad"].head()
df_bad_tool

,question,answer_agent,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning


Simpler model tends to just provide direct answers without considering multiple queries or complex reasoning, which can lead to better tools scores. But answer quality might suffer as a result.

Overall based on the benchmark evaluation, even though simpler models tend to achieve better tools scores due to their direct answers, this often comes at the cost of answer quality. The best model based on the overall balance between tools scores and answer quality appears to be the more complex model that considers multiple queries and engages in deeper reasoning. For that reason, deepseek-v4-flash have the most balanced performance.